# One conflicting map: how much does it move the scores?

[Notebook 06](06_optical_check_s2.ipynb) found two EMS maps of EMSR664 AOI01, less than a day apart, that share 0.5 of 52.1 km². Until now each map was scored apart, so a pixel flooded in one counts as a false alarm against the other.

This notebook repeats the leave-one-activation-out cross-validation of notebook 04 under three label rules, tuning and scoring by the same rule:

* **every map apart**, as so far;
* **any map of the image**: flooded if any map drawn for that radar image says so;
* **without the smaller map**: where an image has two maps, only the one with more flood.

Only maps that share a **radar image** are combined. Maps of one area on different passes (EMSR795, EMSR843, EMSR855) show different days, and their water may really differ.

Whole images are used, not chips, so each pixel of an area counts once per map. **0 PU**: it reads the pairs and references of notebook 02.

In [ ]:
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
import rasterio
from scipy.ndimage import label as connected

from copernicus_ems.dataset import find_pairs
from copernicus_ems.ml import FLOODED, IGNORE

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
KM2 = 0.04**2  # one 40 m pixel
DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-all-maps-s1.png"))


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


pairs = [p for p in find_pairs(DATA_DIR, ACTIVATIONS) if p.reference is not None]
assert pairs, "no pair with a reference: run notebook 02 first"
by_image = defaultdict(list)
for p in pairs:
    by_image[p.image].append(p)

images = []
for image, ps in by_image.items():
    with rasterio.open(image) as src:
        now = src.read()
    with rasterio.open(ps[0].reference) as src:
        before = src.read()
    valid = (now[-1] > 0) & (before[-1] > 0)
    maps = []
    for p in ps:
        with rasterio.open(p.labels) as src:
            labels = src.read(1)
        sensors = sorted({m["sensor"] for m in p.meta.get("made_from", [])})
        maps.append(
            {
                "product": p.product,
                "sensors": sensors,
                "same": "Sentinel-1" in sensors,
                "wet": (labels == FLOODED) & valid,
                "observed": (labels != IGNORE) & valid,
            }
        )
    images.append(
        {"activation": ps[0].activation, "vv": db(now[0]), "pre": db(before[0]), "maps": maps}
    )
shared = [d for d in images if len(d["maps"]) > 1]
print(
    f"{len(pairs)} maps on {len(images)} radar images; {len(shared)} image(s) with more than one map"
)

## 1. The two maps of AOI01

Where each one puts its flood, as a share of the area. If one map's flood stays in a corner of the area, that map may have looked at only part of it, while its label still says dry everywhere else.

In [ ]:
for d in shared:
    for m in d["maps"]:
        rows, cols = np.nonzero(m["wet"])
        height, width = m["wet"].shape
        print(
            f"{m['product']:20} {', '.join(m['sensors']):14} {m['wet'].sum() * KM2:6.1f} km²  "
            f"columns {cols.min() / width:4.0%}-{cols.max() / width:4.0%}, rows {rows.min() / height:4.0%}-{rows.max() / height:4.0%}, "
            f"central 90% of columns {np.percentile(cols, 5) / width:4.0%}-{np.percentile(cols, 95) / width:4.0%}"
        )
    a, b = (m["wet"] for m in d["maps"])
    print(f"shared {(a & b).sum() * KM2:.1f} km², in either {(a | b).sum() * KM2:.1f} km²")

## 2. Three label rules, each tuned and scored the same way

Two detectors of notebook 04: the VV threshold, and VV < *t* dB, VV_pre ≥ *t*ₚ dB, patches of at least *n* pixels. In each fold the values are chosen on the other activations, under the same rule used to score.

In [ ]:
RULES = ["every map apart", "any map of the image", "without the smaller map"]
DETECTORS = ["VV threshold", "not dark before, min size"]
SUBSETS = ["independent", "same image"]
EIGHT = np.ones((3, 3), dtype=bool)


def units(rule):
    out = []
    for d in images:
        maps = d["maps"]
        if rule == RULES[0] or len(maps) == 1:
            out += [(d, m["wet"], m["observed"], m["same"]) for m in maps]
        elif rule == RULES[1]:
            wet = np.any([m["wet"] for m in maps], axis=0)
            observed = np.any([m["observed"] for m in maps], axis=0)
            out.append((d, wet, observed, all(m["same"] for m in maps)))
        else:
            m = max(maps, key=lambda m: m["wet"].sum())
            out.append((d, m["wet"], m["observed"], m["same"]))
    return out


def min_size(predicted, n):
    if n <= 1:
        return predicted
    labelled, _ = connected(predicted, structure=EIGHT)
    big = np.bincount(labelled.ravel()) >= n
    big[0] = False
    return big[labelled]


def tally(predicted, wet, observed):
    return np.array(
        [(predicted & wet).sum(), (~predicted & wet).sum(), (predicted & observed & ~wet).sum()],
        dtype=np.int64,
    )


def csi(c):
    return c[0] / max(c.sum(), 1)


def tune(train):
    vv = np.concatenate([u[0]["vv"][u[2]] for u in train])
    pre = np.concatenate([u[0]["pre"][u[2]] for u in train])
    wet = np.concatenate([u[1][u[2]] for u in train])

    def score(predicted):
        hits = (predicted & wet).sum()
        return hits / max(wet.sum() + (predicted & ~wet).sum(), 1)

    t0 = max(np.arange(-22, -10, 0.5), key=lambda t: score(vv < t))
    t, tp = max(
        [(a, b) for a in np.arange(-22, -10, 0.5) for b in np.arange(-22, -8, 1.0)],
        key=lambda c: score((vv < c[0]) & (pre >= c[1])),
    )
    n = max(
        (1, 2, 4, 8, 16, 32),
        key=lambda k: csi(
            sum(
                tally(min_size((u[0]["vv"] < t) & (u[0]["pre"] >= tp), k), u[1], u[2])
                for u in train
            )
        ),
    )
    return t0, t, tp, n


pooled, per_activation = {}, {}
FOLDS = sorted({d["activation"] for d in images})
for rule in RULES:
    all_units = units(rule)
    for a in FOLDS:
        t0, t, tp, n = tune([u for u in all_units if u[0]["activation"] != a])
        print(f"{rule:24} {a}: VV < {t0:.1f} | VV < {t:.1f}, VV_pre >= {tp:.0f}, >= {n} px")
        for d, wet, observed, same in (u for u in all_units if u[0]["activation"] == a):
            subset = SUBSETS[1] if same else SUBSETS[0]
            found = {
                DETECTORS[0]: d["vv"] < t0,
                DETECTORS[1]: min_size((d["vv"] < t) & (d["pre"] >= tp), n),
            }
            for name, predicted in found.items():
                c = tally(predicted, wet, observed)
                pooled[rule, name, subset] = pooled.get((rule, name, subset), 0) + c
                if name == DETECTORS[1]:
                    key = (rule, a, subset)
                    per_activation[key] = per_activation.get(key, 0) + c

## 3. Scores, every flood counted once

In [ ]:
print(
    f"{'':26} {'rule':24} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'bias':>6} {'mapped km²':>11} {'false km²':>10}"
)
for name in DETECTORS:
    for subset in SUBSETS:
        for rule in RULES:
            hits, misses, false = pooled[rule, name, subset]
            print(
                f"{name:26} {rule:24} {subset:12} {hits / (hits + misses + false):6.3f} {hits / (hits + misses):6.3f} "
                f"{false / max(hits + false, 1):6.3f} {(hits + false) / (hits + misses):6.2f} {(hits + misses) * KM2:11.1f} {false * KM2:10.1f}"
            )

print("\nCSI of 'not dark before, min size' per activation")
print(f"{'':10} {'labels':12}" + "".join(f"{r:>26}" for r in RULES))
for a in FOLDS:
    for subset in SUBSETS:
        if (RULES[0], a, subset) in per_activation:
            print(
                f"{a:10} {subset:12}"
                + "".join(f"{csi(per_activation[r, a, subset]):26.3f}" for r in RULES)
            )

## 4. Figure

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"independent": "#2a78d6", "same image": "#eb6834"}
fig, ax = plt.subplots(figsize=(8, 3.2))
height = 0.36
for k, subset in enumerate(SUBSETS):
    y = np.arange(len(RULES)) + (k - 0.5) * (height + 0.04)
    values = [csi(pooled[r, DETECTORS[1], subset]) for r in RULES]
    ax.barh(y, values, height=height, color=COLORS[subset], label=f"{subset} labels")
    for yy, v in zip(y, values, strict=True):
        ax.text(v + 0.005, yy, f"{v:.2f}", va="center", fontsize=9, color="#444")
ax.set_yticks(np.arange(len(RULES)), RULES)
ax.invert_yaxis()
ax.set_xlim(0, 0.55)
ax.set_xlabel("CSI, not dark before and min size, leave-one-activation-out")
ax.grid(axis="x", color="#e5e5e5")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **One map moves the pooled score.** Only one radar image of 23 has two maps. Combining them or dropping the smaller lifts the independent CSI from 0.17 to 0.23–0.24 and halves the false area (128 to 53 km²). The other activations barely move.
* **The smaller map may be partial.** Its flood stays in a small part of AOI01, while its label says dry everywhere else in the area. That fits a map that looked at only part of the area, but this test does not prove it.
* **Now misses matter more.** With bias 0.66 and POD 0.31, the detector finds less than the maps show. Water that receded or lay under vegetation could explain that; these scores cannot tell which.
* **Same-image scores do not change**, since no same-image label is on the shared radar image.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*